Using A100 GPU

# SDXL

In [ ]:
!pip install --upgrade pip
!pip install --pre xformers -f https://download.pytorch.org/whl/cu118/torch_stable.html
!pip install pytorch-lightning
!pip install torch torchvision torchaudio --upgrade
!pip install diffusers transformers accelerate safetensors pytorch-lightning
!pip install torch_kmeans
import torch
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate safetensors
!pip install numpy matplotlib
import sys
import importlib.util
!pip install nltk
import nltk
import random

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
!git clone https://github.com/jfung07/bounded-attention.git
%cd bounded-attention

In [ ]:
!pip install -r requirements.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
import torch
from diffusers.models.attention_processor import AttnProcessor


class BoundedAttentionProcessor(AttnProcessor):
    def __init__(self, boxes, token_indices, image_size=1024):
        """
        boxes: list of [x1, y1, x2, y2] normalized (0–1)
        token_indices: dict mapping box_id -> list of token indices
        """
        super().__init__()
        self.boxes = boxes
        self.token_indices = token_indices
        self.image_size = image_size

    def __call__(
        self,
        attn,
        hidden_states,
        encoder_hidden_states=None,
        attention_mask=None,
        temb=None,
    ):
        batch_size, seq_len, dim = hidden_states.shape
        if encoder_hidden_states is None:
          return hidden_states

        # Q, K, V
        query = attn.to_q(hidden_states)
        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        query = attn.head_to_batch_dim(query)
        key = attn.head_to_batch_dim(key)
        value = attn.head_to_batch_dim(value)

        # Attention scores
        attention_scores = torch.bmm(query, key.transpose(-1, -2))
        attention_scores *= attn.scale

        # Apply bounded mask
        attention_scores = self.apply_bounded_mask(
            attention_scores,
            hidden_states.shape[1],
            encoder_hidden_states.shape[1],
            hidden_states.device
        )
        attention_probs = attention_scores.softmax(dim=-1)
        hidden_states = torch.bmm(attention_probs, value)

        hidden_states = attn.batch_to_head_dim(hidden_states)
        hidden_states = attn.to_out[0](hidden_states)
        hidden_states = attn.to_out[1](hidden_states)

        return hidden_states

    def apply_bounded_mask(self, scores, spatial_tokens, text_tokens, device):
        """
        scores: (B*heads, spatial_tokens, text_tokens)
        """

        # Infer spatial resolution
        h = w = int(spatial_tokens ** 0.5)
        if h * w != spatial_tokens:
          return scores

        mask = torch.zeros_like(scores)

        for box_id, box in enumerate(self.boxes):
            x1, y1, x2, y2 = box

            # Scale box to feature map resolution
            x1 = int(x1 * w)
            x2 = int(x2 * w)
            y1 = int(y1 * h)
            y2 = int(y2 * h)

            x1, x2 = max(0, x1), min(w, x2)
            y1, y2 = max(0, y1), min(h, y2)

            region = torch.zeros(h, w, device=device)
            region[y1:y2, x1:x2] = 1
            region = region.flatten()

            token_ids = self.token_indices[box_id]

            for t in token_ids:
              region_exp = region.unsqueeze(0).expand(scores.shape[0], -1)
              mask[:, :, t] = region_exp

        # Large negative value to block
        mask_value = -1.5  # much softer than 5
        scores = scores + (mask - 1) * mask_value

        return scores

## Shark/Dolphin

In [ ]:
from transformers import CLIPTokenizer
prompt = "A dolphin being chased by a shark in the ocean"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

In [ ]:
experiments = [
    # baseline
    {"prompt": "A shark chasing a dolphin in the ocean.", "token_map": [[2], [5]]},
    # conversational/polite(formalities at start)
    {"prompt": "Please show me an image of a shark chasing a dolphin in the ocean.", "token_map": [[8], [11]]},
    # direct only key words
    {"prompt": "Shark chase dolphin, ocean.", "token_map": [[1], [3]]},
    # high detail
    {"prompt": "A shark with its mouth open, swimming behind and chasing a smiling dolphin in the blue ocean.", "token_map": [[2,3,4,5,6], [13,14]]},
    # environment first
    {"prompt": "In the ocean, a shark chasing a dolphin.", "token_map": [[6], [9]]},
    # conversational/polite(formalities at end) and question form
    {"prompt": "A shark chasing a dolphin in the ocean, can you show me an image of that?", "token_map": [[2], [5]]},
    # second subject referenced first
    {"prompt": "A dolphin being chased by a shark in the ocean", "token_map": [[7], [2]]},
]

In [ ]:
boxes = [
    [0.1, 0.2, 0.45, 0.8],  # shark
    [0.55, 0.2, 0.9, 0.8]   # dolphin
]

In [ ]:
from IPython.display import display
import os
from PIL import ImageDraw
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(256)
np.random.seed(256)

for i, exp in enumerate(experiments):
  processor = BoundedAttentionProcessor(boxes, exp['token_map'])
  for name, module in pipe.unet.named_modules():
      if name.endswith("attn2"):
          module.set_processor(processor)
  prompt = exp['prompt']
  tokens = pipe.tokenizer(
      prompt,
      return_tensors="pt"
  )
  tokens = pipe.tokenizer(prompt, return_tensors="pt")
  print(tokens["input_ids"])
  image = pipe(
      prompt,
      num_inference_steps=30,
      guidance_scale=7.5,
  ).images[0]
  draw = ImageDraw.Draw(image)

  width, height = image.size

  for box in boxes:
      x_min, y_min, x_max, y_max = box

      # Convert normalized coords (0-1) to pixel coords
      x0 = int(x_min * width)
      y0 = int(y_min * height)
      x1 = int(x_max * width)
      y1 = int(y_max * height)

      draw.rectangle(
          [(x0, y0), (x1, y1)],
          outline="red",
          width=4
      )
  filename = f"outputs/exp_{i}.png"
  image.save(filename)

In [ ]:
import shutil

# Replace 'my_folder' with your folder name
shutil.make_archive('outputs', 'zip', 'outputs')
from google.colab import files
files.download('outputs.zip')

## Plants

In [ ]:
from transformers import CLIPTokenizer
prompt = "Tulips in a porcelain pot and orchids in a metal can and sunflowers in a glass jar"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

In [ ]:
experiments = [
    # baseline
    {"prompt": "A porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers.", "token_map": [[2,3], [5], [8,9], [11], [14,15], [17]]},
    # conversational/polite(formalities at start)
    {"prompt": "Please show me an image of a porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers.", "token_map": [[8,9], [11], [14,15], [17], [20, 21], [23]]},
    # direct only key words
    {"prompt": "Porcelain pot tulips, metal can orchids, glass jar sunflowers.", "token_map": [[1,2], [3], [5,6], [7], [9,10], [11]]},
    # high detail
    {"prompt": "A white porcelain pot with pink tulips and a silver metal can with purple orchids and a clear glass jar with yellow sunflowers, next to each other on a wooden table.", "token_map": [[2,3,4], [6,7], [10, 11, 12], [14, 15], [18, 19, 20], [22, 23]]},
    # environment first
    {"prompt": "In the kitchen, a porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers.", "token_map": [[6,7], [9], [12,13], [15], [18, 19], [21]]},
    # conversational/polite(formalities at end) and question form
    {"prompt": "A porcelain pot with tulips and a metal can with orchids and a glass jar with sunflowers, can you show me an image of that?", "token_map": [[2,3], [5], [8,9], [11], [14,15], [17]]},
    # second subject referenced first
    {"prompt": "Tulips in a porcelain pot and orchids in a metal can and sunflowers in a glass jar.", "token_map": [[4,5], [1], [10,11], [7], [16,17], [13]]},
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Example image (replace with your own image)
image = np.ones((500, 800, 3))  # white background

# Normalized bounding boxes
boxes = [
    [0.1, 0.5, 0.25, 0.85],     # pot1
    [0.05, 0.05, 0.3, 0.5],     # flower1
    [0.45, 0.45, 0.6, 0.8],     # pot2
    [0.37, 0.15, 0.65, 0.45],   # flower2
    [0.75, 0.65, 0.9, 0.95],    # pot3
    [0.7, 0.1, 0.95, 0.65]      # flower3
]

fig, ax = plt.subplots(1)
ax.imshow(image)

img_h, img_w = image.shape[:2]

for box in boxes:
    x_min, y_min, x_max, y_max = box

    # Convert normalized coordinates to pixel values
    x = x_min * img_w
    y = y_min * img_h
    width = (x_max - x_min) * img_w
    height = (y_max - y_min) * img_h

    rect = patches.Rectangle(
        (x, y),
        width,
        height,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

plt.axis('off')
plt.show()

In [ ]:
boxes = [
    [0.1, 0.5, 0.25, 0.85],     # pot1
    [0.05, 0.05, 0.3, 0.5],     # flower1
    [0.45, 0.45, 0.6, 0.8],     # pot2
    [0.37, 0.15, 0.65, 0.45],   # flower2
    [0.75, 0.65, 0.9, 0.95],    # pot3
    [0.7, 0.1, 0.95, 0.65]      # flower3
]

In [ ]:
from IPython.display import display
import os
from PIL import ImageDraw
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(256)
np.random.seed(256)

for i, exp in enumerate(experiments):
  processor = BoundedAttentionProcessor(boxes, exp['token_map'])
  for name, module in pipe.unet.named_modules():
      if name.endswith("attn2"):
          module.set_processor(processor)
  prompt = exp['prompt']
  tokens = pipe.tokenizer(
      prompt,
      return_tensors="pt"
  )
  tokens = pipe.tokenizer(prompt, return_tensors="pt")
  print(tokens["input_ids"])
  image = pipe(
      prompt,
      num_inference_steps=30,
      guidance_scale=7.5,
  ).images[0]
  draw = ImageDraw.Draw(image)

  width, height = image.size

  for box in boxes:
      x_min, y_min, x_max, y_max = box

      # Convert normalized coords (0-1) to pixel coords
      x0 = int(x_min * width)
      y0 = int(y_min * height)
      x1 = int(x_max * width)
      y1 = int(y_max * height)

      draw.rectangle(
          [(x0, y0), (x1, y1)],
          outline="red",
          width=4
      )
  filename = f"outputs/exp_{i}.png"
  image.save(filename)

In [ ]:
import shutil

# Replace 'my_folder' with your folder name
shutil.make_archive('outputs', 'zip', 'outputs')
from google.colab import files
files.download('outputs.zip')

## Cartoon People

In [ ]:
from transformers import CLIPTokenizer
prompt = "3D Pixar animation of a fairy, an elf, and a princess in a castle"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

In [ ]:
experiments = [
    # baseline
    {"prompt": "3D Pixar animation of a princess, an elf, and a fairy in a castle.", "token_map": [[7], [10], [14]]},
    # conversational/polite(formalities at start)
    {"prompt": "Please show me a 3D Pixar animation of a princess, an elf, and a fairy in a castle.", "token_map": [[11], [14], [18]]},
    # direct only key words
    {"prompt": "3D Pixar animation, princess, elf, fairy, castle", "token_map": [[6], [8], [10]]},
    # high detail
    {"prompt": "A 3D Pixar animation of a blonde princess wearing a pink dress next to a fat elf with pointy ears with a tan fairy wearing a green dress in a diamond castle.", "token_map": [[8,9,10,12,13], [17,18,19,20,21], [24,25,26,27,28,29]]},
    # environment first
    {"prompt": "In a castle, a princess, an elf, and a fairy 3D Pixar animation.", "token_map": [[6], [9], [13]]},
    # conversational/polite(formalities at end) and question form
    {"prompt": "A princess, an elf, and a fairy in a castle, can you show me a 3D Pixar animation of that?", "token_map": [[2], [5], [9]]},
    # second subject referenced first
    {"prompt": "3D Pixar animation of a fairy, an elf, and a princess in a castle.", "token_map": [[14], [10], [7]]},
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Example image (replace with your own image)
image = np.ones((500, 800, 3))  # white background

# Normalized bounding boxes
boxes = [
    [0.1, 0.1, 0.3, 0.85],      # princess
    [0.4, 0.45, 0.65, 0.85],     # elf
    [0.7, 0.1, 0.85, 0.35]      # fairy
]

fig, ax = plt.subplots(1)
ax.imshow(image)

img_h, img_w = image.shape[:2]

for box in boxes:
    x_min, y_min, x_max, y_max = box

    # Convert normalized coordinates to pixel values
    x = x_min * img_w
    y = y_min * img_h
    width = (x_max - x_min) * img_w
    height = (y_max - y_min) * img_h

    rect = patches.Rectangle(
        (x, y),
        width,
        height,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

plt.axis('off')
plt.show()

In [ ]:
boxes = [
    [0.1, 0.1, 0.3, 0.85],      # princess
    [0.4, 0.45, 0.65, 0.85],     # elf
    [0.7, 0.1, 0.85, 0.35]      # fairy
]

In [ ]:
from IPython.display import display
import os
from PIL import ImageDraw
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(256)
np.random.seed(256)

for i, exp in enumerate(experiments):
  processor = BoundedAttentionProcessor(boxes, exp['token_map'])
  for name, module in pipe.unet.named_modules():
      if name.endswith("attn2"):
          module.set_processor(processor)
  prompt = exp['prompt']
  tokens = pipe.tokenizer(
      prompt,
      return_tensors="pt"
  )
  tokens = pipe.tokenizer(prompt, return_tensors="pt")
  print(tokens["input_ids"])
  image = pipe(
      prompt,
      num_inference_steps=30,
      guidance_scale=7.5,
  ).images[0]
  draw = ImageDraw.Draw(image)

  width, height = image.size

  for box in boxes:
      x_min, y_min, x_max, y_max = box

      # Convert normalized coords (0-1) to pixel coords
      x0 = int(x_min * width)
      y0 = int(y_min * height)
      x1 = int(x_max * width)
      y1 = int(y_max * height)

      draw.rectangle(
          [(x0, y0), (x1, y1)],
          outline="red",
          width=4
      )
  filename = f"outputs/exp_{i}.png"
  image.save(filename)

In [ ]:
import shutil

# Replace 'my_folder' with your folder name
shutil.make_archive('outputs', 'zip', 'outputs')
from google.colab import files
files.download('outputs.zip')

## Real People

In [ ]:
from transformers import CLIPTokenizer
prompt = "japanese girl with black hair and japanese girl with long brown hair presenting a slideshow in a classroom"

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
tokens = tokenizer(prompt)

for i, tok in enumerate(tokens.input_ids):
    print(i, tokenizer.decode([tok]))

In [ ]:
experiments = [
    # baseline
    {"prompt": "two asian girls presenting in a classroom", "token_map": [[2,3], [2,3]]},
]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Example image (replace with your own image)
image = np.ones((500, 800, 3))  # white background

# Normalized bounding boxes
boxes = [
    [0.1, 0.3, 0.3, 0.85],     # elf
    [0.7, 0.3, 0.9, 0.85],     # elf
]

fig, ax = plt.subplots(1)
ax.imshow(image)

img_h, img_w = image.shape[:2]

for box in boxes:
    x_min, y_min, x_max, y_max = box

    # Convert normalized coordinates to pixel values
    x = x_min * img_w
    y = y_min * img_h
    width = (x_max - x_min) * img_w
    height = (y_max - y_min) * img_h

    rect = patches.Rectangle(
        (x, y),
        width,
        height,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

plt.axis('off')
plt.show()

In [ ]:
boxes = [
    [0.1, 0.3, 0.3, 0.85],     # elf
    [0.7, 0.3, 0.9, 0.85],     # elf
]

In [ ]:
from IPython.display import display
import os
from PIL import ImageDraw
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(256)
np.random.seed(256)

for i, exp in enumerate(experiments):
  processor = BoundedAttentionProcessor(boxes, exp['token_map'])
  for name, module in pipe.unet.named_modules():
      if name.endswith("attn2"):
          module.set_processor(processor)
  prompt = exp['prompt']
  tokens = pipe.tokenizer(
      prompt,
      return_tensors="pt"
  )
  tokens = pipe.tokenizer(prompt, return_tensors="pt")
  print(tokens["input_ids"])
  image = pipe(
      prompt,
      num_inference_steps=30,
      guidance_scale=7.5,
  ).images[0]
  draw = ImageDraw.Draw(image)

  width, height = image.size


  filename = f"outputs/exp_{i}.png"
  image.save(filename)

In [ ]:
import shutil

# Replace 'my_folder' with your folder name
shutil.make_archive('outputs', 'zip', 'outputs')
from google.colab import files
files.download('outputs.zip')